In [1]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(
    sys.version.split()[0],
    numpy.__version__,
    pandas.__version__,
    sklearn.__version__,
    matplotlib.__version__
)

3.12.10 2.3.4 2.3.3 1.7.2 3.10.7


In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3101

data = pd.read_csv("data/spambase.csv")

feature_names = [
    column for column in data.columns
    if column != "spam"
]

X = data[feature_names].to_numpy()
y = data["spam"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

print(
    X.shape,
    round(100 * y.mean(), 1),
    X_train.shape,
    X_test.shape
)

assert X.shape == (4204, 57)
assert X_train.shape == (3153, 57)
assert X_test.shape == (1051, 57)

(4204, 57) 39.9 (3153, 57) (1051, 57)


# Task 4 — Random Forests & Ensembles

**Student:** Rakhimbek Baizhan  
**SEED:** 3101 — the last four digits of my student ID.

The dataset contains 4,204 emails described by 57 numerical features.
The target is `spam`: 1 represents spam and 0 represents legitimate email.

I use a stratified 75% training and 25% test split. Cross-validation
comparisons use the same five stratified folds of the training split.

All standard deviations use `ddof=0`. A paired difference is inside
the fold spread when its absolute mean is no greater than its standard
deviation. In that case, I do not declare a winner.

Thresholds, the best voting member and the number of boosting rounds
are selected using training data only.

In [8]:
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    RandomForestClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)

bag = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,
    random_state=SEED
)

rf = RandomForestClassifier(
    n_estimators=300,
    oob_score=True,
    random_state=SEED,
    n_jobs=-1
)

lr = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

bag_lr = BaggingClassifier(
    lr,
    n_estimators=50,
    random_state=SEED
)

knn = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=15)
)

ada = AdaBoostClassifier(
    n_estimators=400,
    random_state=SEED
)

members = [
    ("lr", lr),
    ("knn", knn),
    ("tree", tree)
]

In [9]:
import math
from itertools import combinations

import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)

cv_scores = {}


def score_table(score_dict):
    """Build a table of five fold scores, their mean and population std."""
    table = pd.DataFrame.from_dict(
        score_dict,
        orient="index",
        columns=[f"Fold {i}" for i in range(1, 6)]
    )

    table["Mean"] = [
        np.mean(values) for values in score_dict.values()
    ]

    table["Std"] = [
        np.std(values, ddof=0) for values in score_dict.values()
    ]

    return table


def print_table(table):
    """Print numerical results to six decimal places."""
    print(
        table.to_string(
            float_format=lambda value: f"{value:.6f}"
        )
    )


def compare_scores(label, differences):
    """Interpret paired accuracy differences using the handout rule."""
    mean = np.mean(differences)
    std = np.std(differences, ddof=0)

    if abs(mean) <= std:
        conclusion = "Inside the fold spread: no winner."
    elif mean > 0:
        conclusion = "Outside the spread: the first model performs better."
    else:
        conclusion = "Outside the spread: the second model performs better."

    print(
        f"{label}: mean={mean:.6f}, std={std:.6f}. "
        f"{conclusion}"
    )

## Part 1 — Bagging and Out-of-Bag Estimates

In [10]:
def oob_mask(sample_indices, n_samples):
    """Return a boolean mask of rows absent from the bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    indices = np.asarray(sample_indices, dtype=np.intp)
    mask[indices] = False
    return mask

In [11]:
assert np.array_equal(
    oob_mask([0, 2, 2], 5),
    [False, True, False, True, True]
)

assert np.array_equal(
    oob_mask([], 3),
    [True, True, True]
)

indices = np.array([3, 0])
original_indices = indices.copy()

mask_result = oob_mask(
    sample_indices=indices,
    n_samples=4
)

assert np.array_equal(mask_result, [False, True, True, False])
assert np.array_equal(indices, original_indices)
assert mask_result.dtype == np.bool_
assert mask_result.shape == (4,)

print("All oob_mask checks passed.")

bag.fit(X_train, y_train)

n = len(y_train)
oob_shares = []
votes = np.zeros((n, 2), dtype=int)

for estimator, sample_indices, features in zip(
    bag.estimators_,
    bag.estimators_samples_,
    bag.estimators_features_
):
    mask = oob_mask(sample_indices, n)
    oob_shares.append(mask.mean())

    rows = np.flatnonzero(mask)

    predictions = estimator.predict(
        X_train[mask][:, features]
    ).astype(int)

    votes[rows, predictions] += 1

mean_oob_share = np.mean(oob_shares)
expected_oob_share = (1 - 1 / n) ** n

share_table = pd.DataFrame(
    {
        "Measured mean": [
            mean_oob_share,
            1 - mean_oob_share
        ],
        "Theoretical value": [
            expected_oob_share,
            1 - expected_oob_share
        ]
    },
    index=["Out of bag", "In bag"]
)

print("\nBootstrap row shares:")
print_table(share_table)

assert np.all(votes.sum(axis=1) > 0)

manual_oob_predictions = votes.argmax(axis=1)
manual_oob_accuracy = np.mean(
    manual_oob_predictions == y_train
)

print(f"\nManual OOB accuracy: {manual_oob_accuracy:.12f}")
print(f"Scikit-learn OOB accuracy: {bag.oob_score_:.12f}")

assert manual_oob_accuracy == bag.oob_score_

All oob_mask checks passed.

Bootstrap row shares:
            Measured mean  Theoretical value
Out of bag       0.367913           0.367821
In bag           0.632087           0.632179

Manual OOB accuracy: 0.934031081510
Scikit-learn OOB accuracy: 0.934031081510


In [12]:
part1_models = {
    "tree": tree,
    "bag": bag,
    "rf": rf,
    "lr": lr,
    "bag_lr": bag_lr
}

for name, model in part1_models.items():
    cv_scores[name] = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy"
    )["test_score"]

print("Accuracy on the same five training folds:")
print_table(
    score_table({
        name: cv_scores[name]
        for name in part1_models
    })
)

part1_differences = {
    "bag - tree": cv_scores["bag"] - cv_scores["tree"],
    "rf - tree": cv_scores["rf"] - cv_scores["tree"],
    "bag_lr - lr": cv_scores["bag_lr"] - cv_scores["lr"]
}

print("\nPaired fold differences:")
print_table(score_table(part1_differences))

print()
for label, differences in part1_differences.items():
    compare_scores(label, differences)

Accuracy on the same five training folds:
         Fold 1   Fold 2   Fold 3   Fold 4   Fold 5     Mean      Std
tree   0.890650 0.890650 0.898574 0.906349 0.892063 0.895657 0.006097
bag    0.919176 0.933439 0.925515 0.936508 0.936508 0.930229 0.006834
rf     0.938193 0.938193 0.947702 0.949206 0.944444 0.943548 0.004635
lr     0.930269 0.931854 0.928685 0.950794 0.938095 0.935939 0.008084
bag_lr 0.928685 0.930269 0.930269 0.952381 0.938095 0.935940 0.008853

Paired fold differences:
               Fold 1    Fold 2   Fold 3   Fold 4   Fold 5     Mean      Std
bag - tree   0.028526  0.042789 0.026941 0.030159 0.044444 0.034572 0.007473
rf - tree    0.047544  0.047544 0.049128 0.042857 0.052381 0.047891 0.003075
bag_lr - lr -0.001585 -0.001585 0.001585 0.001587 0.000000 0.000001 0.001418

bag - tree: mean=0.034572, std=0.007473. Outside the spread: the first model performs better.
rf - tree: mean=0.047891, std=0.003075. Outside the spread: the first model performs better.
bag_lr - lr: mea

In [13]:
rf.fit(X_train, y_train)

forest_oob = rf.oob_score_
forest_cv_mean = np.mean(cv_scores["rf"])
forest_cv_std = np.std(cv_scores["rf"], ddof=0)

forest_test_accuracy = rf.score(X_test, y_test)

forest_test_se = np.sqrt(
    forest_test_accuracy
    * (1 - forest_test_accuracy)
    / len(y_test)
)

forest_table = pd.DataFrame(
    {
        "Value": [
            forest_oob,
            forest_cv_mean,
            forest_cv_std,
            forest_test_accuracy,
            forest_test_se
        ]
    },
    index=[
        "OOB accuracy",
        "CV mean accuracy",
        "CV standard deviation",
        "Test accuracy",
        "Test standard error"
    ]
)

print_table(forest_table)

oob_cv_difference = abs(forest_oob - forest_cv_mean)

print(f"\nAbsolute OOB-CV difference: {oob_cv_difference:.6f}")
print(f"Forest CV standard deviation: {forest_cv_std:.6f}")
print(f"OOB and CV agree: {oob_cv_difference <= forest_cv_std}")

for name, estimate in [
    ("OOB", forest_oob),
    ("CV mean", forest_cv_mean)
]:
    difference = abs(estimate - forest_test_accuracy)

    print(
        f"{name}: absolute difference from test={difference:.6f}; "
        f"within one test SE={difference <= forest_test_se}"
    )

                         Value
OOB accuracy          0.947352
CV mean accuracy      0.943548
CV standard deviation 0.004635
Test accuracy         0.942912
Test standard error   0.007157

Absolute OOB-CV difference: 0.003804
Forest CV standard deviation: 0.004635
OOB and CV agree: True
OOB: absolute difference from test=0.004440; within one test SE=True
CV mean: absolute difference from test=0.000636; within one test SE=True


### Discussion

The measured mean OOB share is approximately 0.367913, close to the
theoretical value of 0.367821. The small difference is expected because
the measured mean comes from 100 random bootstrap samples. The in-bag
share counts distinct rows seen at least once, not the total number
of bootstrap draws.

My manual OOB accuracy is 0.934031081510, exactly equal to
`bag.oob_score_`. Each row receives votes only from trees that did not
use it for training, and a tied vote selects class 0.

Bagging improves the decision tree's mean CV accuracy from 0.895657
to 0.930229. The paired improvement is 0.034572 with a standard deviation
of 0.007473, so it is outside the fold spread. A fully grown tree has
high variance, and combining bootstrap-trained trees reduces this
variability.

For logistic regression, the paired mean difference is approximately
0.000001 with a standard deviation of 0.001418. This is inside the fold
spread, so there is no winner. Regularized logistic regression is more
stable than a fully grown tree, leaving less variance for bagging to
reduce.

The random forest improves over the tree by 0.047891 on average, with
a paired standard deviation of 0.003075. Random feature subsets help
reduce correlation between trees, in addition to the benefit of averaging.

Forest OOB accuracy is 0.947352, while CV accuracy is 0.943548 with a
standard deviation of 0.004635. Their absolute difference, 0.003804,
is smaller than the CV standard deviation. Therefore, OOB is a fair
stand-in for CV here under the assignment's rule.

Test accuracy is 0.942912 with a standard error of 0.007157. OOB and CV
are both slightly higher, but both differences are within one test
standard error. This comparison is descriptive: test SE does not
represent all uncertainty in OOB and CV.